# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We'll inspect the record sets, list the available `@id` for each, and display the available fields.


In [ ]:
# List available record sets and their field IDs.
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"RecordSet @id: {rs.id}")
        record_set_ids.append(rs.id)
        # Show field IDs in this record set
        print("Available fields:")
        for field in rs.fields:
            print(f"  Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        print()else:
    print("No record sets found in dataset metadata.")

# If no record_set_ids found, try fetching by fallback: 
if not record_set_ids:
    # Try to infer record set IDs from the dataset object (for some schemas)
    try:
        record_set_ids = dataset.record_set_ids
        print("Fetched record_set_ids from dataset:", record_set_ids)
    except Exception as e:
        print("Could not determine record set IDs automatically.")


## 3. Data Extraction
Load data from each available record set into a pandas DataFrame for analysis. We reference record sets and fields by their `@id` only.


In [ ]:
# Set the list of record set IDs for data extraction:
# Please edit/edit the cell above ('Data Overview') to get the actual record_set_ids and field IDs.
if not record_set_ids:
    # For this dataset, likely record set id is 'records:MainTable' based on schema conventions, or similar. Use known logic or comment if the previous cell listed them.
    record_set_ids = [rs.id for rs in getattr(metadata, 'record_sets', [])]

if not record_set_ids:
    # Try to use single fallback for Croissant datasets with a single table
    record_set_ids = ['cr:MainTable']

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Successfully loaded records for {record_set_id}")
            print(f"Fields: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records found for {record_set_id}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Let's perform simple data processing operations, including filtering numeric fields, normalization, grouping, and categorization. 
We reference all columns by their `@id` only, as shown in the previous steps.


In [ ]:
# Select the record set and choose a numeric field by @id
from IPython.display import display

# Auto-select numeric field column and group field from one of the loaded DataFrames.
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Heuristically pick a numeric field: look for a column like 'cr:abundance' or with float/int values
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if not numeric_field_candidates:
        # Try to cast some likely columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception: pass
        numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        numeric_field = df.columns[0]
        print(f"Defaulting to first field: {numeric_field}")
    # Choose a group field, e.g., 'cr:sample' or one that might be categorical
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field:
            group_field = col
            break
    if not group_field:
        group_field = df.columns[0]
    print(f"Grouping by field: {group_field}")
    # Outlier filtering (e.g., values > df[numeric_field].quantile(0.95))
    threshold = df[numeric_field].quantile(0.95) if (df[numeric_field].dtype in ['float64','int64'] and df[numeric_field].count()>10) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")
    display(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group-by analysis (means)
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No dataframes were loaded in previous step.")

## 5. Visualization
Visualize the distribution of the numeric field and its relationship to the group field, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[list(dataframes.keys())[0]]
    # Draw histogram for the numeric field
    if numeric_field in df.columns:
        plt.figure(figsize=(7,5))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.grid(True)
        plt.show()
    # Boxplot by group field
    if group_field and group_field in df.columns and numeric_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} across {group_field}")
        plt.show()


## 6. Conclusion
In this notebook we loaded mass spectrometry protein and peptide data from the FAIR^2 Croissant schema, explored available record sets and fields using their `@id`, filtered and normalized numeric attributes, and visualized distributions and groupings.

* Please refer to the dataset documentation for more details on each field and best practices in mass spectrometry or protein bioinformatics analysis for further steps.
